In [1]:
import pandas as pd
import numpy as np
# Load dataset
dataset = pd.read_csv("CKD.csv")

# Normalize column names
dataset.columns = dataset.columns.str.strip().str.lower()

# Clean & encode target
dataset['classification'] = dataset['classification'].astype(str).str.strip().str.lower()
dataset = dataset[dataset['classification'].isin(['yes', 'no'])]
dataset['classification'] = dataset['classification'].map({'yes': 1, 'no': 0})

# Separate features and target
independant = dataset.drop("classification", axis=1)
dependant = dataset["classification"]

# Identify feature types
numeric_features = independant.select_dtypes(include=["int64", "float64"]).columns
categorical_features = independant.select_dtypes(include=["object"]).columns

# Preprocessing Numerical and Categorical data
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# Pre-Process Nominal data using one hot encoding
from sklearn.preprocessing import OneHotEncoder
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Pre-Process Numerical and Categorical data
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])
#print(preprocessor)



In [2]:
# Executing the output data using pipeline and Decision Tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV

pipeline = Pipeline([ ("preprocessor", preprocessor), ("classifier", DecisionTreeClassifier(random_state=42))])

# Grid Search parameters for Decision Tree
param_grid = {"classifier__criterion": 
              ["gini", "entropy"],"classifier__max_depth": [None, 5, 10, 20, 30],
                "classifier__min_samples_split": [2, 5, 10], "classifier__min_samples_leaf": [1, 2, 5],
              "classifier__class_weight": [None, "balanced"]}

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(independant,dependant,test_size=0.33,random_state=42,
                                                    stratify=dependant)

# Grid Search - Creating and fitting the model
grid_search = GridSearchCV(estimator=pipeline,param_grid=param_grid,cv=5,scoring="accuracy", n_jobs=-1)

grid_search.fit(X_train, y_train)

# Best model results
print("Best Parameters:")
print(grid_search.best_params_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

Best Parameters:
{'classifier__class_weight': 'balanced', 'classifier__criterion': 'entropy', 'classifier__max_depth': None, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2}


In [3]:
##Printing Accuracy , Confusion Matric and Classification Report
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy=accuracy_score(y_test, y_pred)
print("\nAccuracy:", accuracy)

cm=confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n",cm)

creport=classification_report(y_test, y_pred)
print("\nClassification Report:\n", creport)


Accuracy: 0.9696969696969697

Confusion Matrix:
 [[48  2]
 [ 2 80]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.96      0.96        50
           1       0.98      0.98      0.98        82

    accuracy                           0.97       132
   macro avg       0.97      0.97      0.97       132
weighted avg       0.97      0.97      0.97       132

